In [ ]:
# ==========================================================
# IMPORTATION DES BIBLIOTHÈQUES
# ==========================================================

import numpy as np
from collections import Counter


# ==========================================================
# CALCUL DE L'ENTROPIE
# ==========================================================

def calcul_entropie(y):
    """
    Cette fonction calcule le degré de désordre des données.
    Plus les classes sont mélangées, plus l'entropie est élevée.
    """

    # Compter le nombre d'exemples de chaque classe
    classes = np.bincount(y)

    # Calcul des probabilités
    probabilites = classes / len(y)

    # Calcul de l'entropie
    entropie = -sum(
        p * np.log2(p)
        for p in probabilites
        if p > 0
    )

    return entropie


# ==========================================================
# CLASSE NOEUD
# ==========================================================

class Noeud:

    def __init__(self,
                 attribut=None,
                 seuil=None,
                 gauche=None,
                 droite=None,
                 valeur=None):

        # Attribut choisi pour la séparation
        self.attribut = attribut

        # Valeur du seuil
        self.seuil = seuil

        # Sous-arbre gauche
        self.gauche = gauche

        # Sous-arbre droit
        self.droite = droite

        # Classe prédite si feuille
        self.valeur = valeur


# ==========================================================
# CLASSE ARBRE DE DÉCISION
# ==========================================================

class ArbreDecision:

    def __init__(self,
                 profondeur_max=4,
                 min_echantillons=2):

        # Profondeur maximale
        self.profondeur_max = profondeur_max

        # Nombre minimal d'observations
        self.min_echantillons = min_echantillons

        # Racine de l'arbre
        self.racine = None


    # ------------------------------------------------------

    def entrainer(self, X, y):

        # Construire l'arbre
        self.racine = self.construire_arbre(X, y)


    # ------------------------------------------------------

    def construire_arbre(self, X, y, profondeur=0):

        # Nombre d'exemples et de caractéristiques
        nb_lignes, nb_colonnes = X.shape

        # Nombre de classes
        nb_classes = len(np.unique(y))

        # Conditions d'arrêt
        if (profondeur >= self.profondeur_max or
            nb_classes == 1 or
            nb_lignes < self.min_echantillons):

            # Classe majoritaire
            classe = Counter(y).most_common(1)[0][0]

            return Noeud(valeur=classe)

        # Recherche du meilleur attribut
        meilleur_attribut, meilleur_seuil = self.meilleur_split(X, y)

        # Séparer les données
        gauche, droite = self.diviser(
            X[:, meilleur_attribut],
            meilleur_seuil
        )

        # Construire les sous-arbres
        branche_gauche = self.construire_arbre(
            X[gauche],
            y[gauche],
            profondeur + 1
        )

        branche_droite = self.construire_arbre(
            X[droite],
            y[droite],
            profondeur + 1
        )

        # Retourner le noeud
        return Noeud(
            meilleur_attribut,
            meilleur_seuil,
            branche_gauche,
            branche_droite
        )


    # ------------------------------------------------------

    def meilleur_split(self, X, y):

        meilleur_gain = -1

        meilleur_attribut = None

        meilleur_seuil = None

        # Tester chaque caractéristique
        for colonne in range(X.shape[1]):

            # Valeurs possibles
            valeurs = np.unique(X[:, colonne])

            # Tester chaque seuil
            for seuil in valeurs:

                gain = self.gain_information(
                    y,
                    X[:, colonne],
                    seuil
                )

                if gain > meilleur_gain:

                    meilleur_gain = gain
                    meilleur_attribut = colonne
                    meilleur_seuil = seuil

        return meilleur_attribut, meilleur_seuil


    # ------------------------------------------------------

    def gain_information(self, y, colonne, seuil):

        # Entropie avant séparation
        entropie_parent = calcul_entropie(y)

        # Diviser les données
        gauche, droite = self.diviser(colonne, seuil)

        # Vérifier si une branche est vide
        if len(gauche) == 0 or len(droite) == 0:
            return 0

        total = len(y)

        nb_gauche = len(gauche)
        nb_droite = len(droite)

        # Calcul des entropies
        entropie_gauche = calcul_entropie(y[gauche])
        entropie_droite = calcul_entropie(y[droite])

        # Entropie pondérée
        entropie_enfants = (
            (nb_gauche / total) * entropie_gauche +
            (nb_droite / total) * entropie_droite
        )

        # Gain d'information
        return entropie_parent - entropie_enfants


    # ------------------------------------------------------

    def diviser(self, colonne, seuil):

        # Indices du groupe gauche
        gauche = np.argwhere(colonne <= seuil).flatten()

        # Indices du groupe droit
        droite = np.argwhere(colonne > seuil).flatten()

        return gauche, droite


    # ------------------------------------------------------

    def predire(self, X):

        # Retourner les prédictions
        return np.array([
            self.parcourir(x, self.racine)
            for x in X
        ])


    # ------------------------------------------------------

    def parcourir(self, x, noeud):

        # Si feuille
        if noeud.valeur is not None:
            return noeud.valeur

        # Aller à gauche
        if x[noeud.attribut] <= noeud.seuil:
            return self.parcourir(x, noeud.gauche)

        # Sinon aller à droite
        return self.parcourir(x, noeud.droite)


# ==========================================================
# EXEMPLE : DIAGNOSTIC MÉDICAL
# ==========================================================

# Chaque ligne représente un patient
#
# Colonne 1 : Température (°C)
# Colonne 2 : Toux
#              0 = Non
#              1 = Oui

X = np.array([
    [36.5, 0],
    [37.0, 0],
    [38.5, 1],
    [39.0, 1],
    [36.8, 0],
    [38.2, 1],
    [39.5, 1],
    [37.2, 0]
])

# Classe cible
# 0 = Sain
# 1 = Malade

y = np.array([
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    0
])

# Création du modèle
modele = ArbreDecision(
    profondeur_max=3,
    min_echantillons=2
)

# Entraîner l'arbre
modele.entrainer(X, y)

# ==========================================================
# TEST SUR UN NOUVEAU PATIENT
# ==========================================================

# Température = 38.7°C
# Toux = Oui

nouveau_patient = np.array([
    [38.7, 1]
])

# Prédiction
prediction = modele.predire(nouveau_patient)

# ==========================================================
# AFFICHAGE DU RÉSULTAT
# ==========================================================

print("===== Diagnostic Médical =====")

print("Température :", nouveau_patient[0][0], "°C")
print("Toux :", "Oui" if nouveau_patient[0][1] == 1 else "Non")

if prediction[0] == 1:
    print("Diagnostic : Le patient est malade.")
else:
    print("Diagnostic : Le patient est sain.")

===== Diagnostic Médical =====
Température : 38.7 °C
Toux : Oui
Diagnostic : Le patient est malade.
